# 第9章: 事前学習済み言語モデル（BERT型）

本章では、BERT型の事前学習済みモデルを利用して、マスク単語の予測や文ベクトルの計算、評判分析器（ポジネガ分類器）の構築に取り組む。

## 80. トークン化

"The movie was full of incomprehensibilities."という文をトークンに分解し、トークン列を表示せよ。

In [29]:
!pip install transformers torch -q

In [30]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "The movie was full of incomprehensibilities."
tokens = tokenizer.tokenize(text)

print(tokens)

['the', 'movie', 'was', 'full', 'of', 'inc', '##omp', '##re', '##hen', '##si', '##bilities', '.']


## 81. マスクの予測

"The movie was full of [MASK]."の"[MASK]"を埋めるのに最も適切なトークンを求めよ。

In [32]:
from transformers import pipeline

# Masked Language Model
unmasker = pipeline(
    "fill-mask",
    model="bert-base-uncased"
)

text = "The movie was full of [MASK]."

results = unmasker(text)

# 最上位候補
print("Best token:", results[0]["token_str"])
print("Score:", results[0]["score"])
print("Sentence:", results[0]["sequence"])

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Best token: fun
Score: 0.10711938142776489
Sentence: the movie was full of fun.


## 82. マスクのtop-k予測

"The movie was full of [MASK]."の"[MASK]"に埋めるのに適切なトークン上位10個と、その確率（尤度）を求めよ。

In [33]:
results = unmasker(text, top_k=10)

print(f"Input: {text}\n")

for rank, result in enumerate(results, start=1):
    print(f"{rank:2d}. Token      : {result['token_str']}")
    print(f"    Probability: {result['score']:.6f}")
    print(f"    Sentence   : {result['sequence']}")
    print()

Input: The movie was full of [MASK].

 1. Token      : fun
    Probability: 0.107119
    Sentence   : the movie was full of fun.

 2. Token      : surprises
    Probability: 0.066345
    Sentence   : the movie was full of surprises.

 3. Token      : drama
    Probability: 0.044684
    Sentence   : the movie was full of drama.

 4. Token      : stars
    Probability: 0.027217
    Sentence   : the movie was full of stars.

 5. Token      : laughs
    Probability: 0.025413
    Sentence   : the movie was full of laughs.

 6. Token      : action
    Probability: 0.019517
    Sentence   : the movie was full of action.

 7. Token      : excitement
    Probability: 0.019038
    Sentence   : the movie was full of excitement.

 8. Token      : people
    Probability: 0.018290
    Sentence   : the movie was full of people.

 9. Token      : tension
    Probability: 0.015031
    Sentence   : the movie was full of tension.

10. Token      : music
    Probability: 0.014646
    Sentence   : the movi

## 83. CLSトークンによる文ベクトル

以下の文の全ての組み合わせに対して、最終層の[CLS]トークンの埋め込みベクトルを用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


In [34]:
import torch
import itertools
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

cls_vectors = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    cls_vector = outputs.last_hidden_state[0, 0, :].numpy()
    cls_vectors.append(cls_vector)

for (s1, v1), (s2, v2) in itertools.combinations(
    zip(sentences, cls_vectors), 2
):
    sim = cosine_similarity([v1], [v2])[0][0]

    print(f"文1: {s1}")
    print(f"文2: {s2}")
    print(f"コサイン類似度: {sim:.4f}")
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


文1: The movie was full of fun.
文2: The movie was full of excitement.
コサイン類似度: 0.9881

文1: The movie was full of fun.
文2: The movie was full of crap.
コサイン類似度: 0.9558

文1: The movie was full of fun.
文2: The movie was full of rubbish.
コサイン類似度: 0.9475

文1: The movie was full of excitement.
文2: The movie was full of crap.
コサイン類似度: 0.9541

文1: The movie was full of excitement.
文2: The movie was full of rubbish.
コサイン類似度: 0.9487

文1: The movie was full of crap.
文2: The movie was full of rubbish.
コサイン類似度: 0.9807



## 84. 平均による文ベクトル

以下の文の全ての組み合わせに対して、最終層の埋め込みベクトルの平均を用いてコサイン類似度を求めよ。

- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."

In [35]:
!pip install transformers torch scikit-learn -q

import torch
import itertools
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish."
]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

sentence_vectors = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    # 最終層の出力
    last_hidden_state = outputs.last_hidden_state

    # 全トークンの平均
    mean_vector = last_hidden_state.mean(dim=1).squeeze().numpy()

    sentence_vectors.append(mean_vector)

print("=== コサイン類似度 ===\n")

for (s1, v1), (s2, v2) in itertools.combinations(
    zip(sentences, sentence_vectors), 2
):
    sim = cosine_similarity([v1], [v2])[0][0]

    print(f"文1: {s1}")
    print(f"文2: {s2}")
    print(f"コサイン類似度: {sim:.4f}")
    print()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== コサイン類似度 ===

文1: The movie was full of fun.
文2: The movie was full of excitement.
コサイン類似度: 0.9568

文1: The movie was full of fun.
文2: The movie was full of crap.
コサイン類似度: 0.8490

文1: The movie was full of fun.
文2: The movie was full of rubbish.
コサイン類似度: 0.8169

文1: The movie was full of excitement.
文2: The movie was full of crap.
コサイン類似度: 0.8352

文1: The movie was full of excitement.
文2: The movie was full of rubbish.
コサイン類似度: 0.7938

文1: The movie was full of crap.
文2: The movie was full of rubbish.
コサイン類似度: 0.9226



## 85. データセットの準備

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) から訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、さらに全てのテキストはトークン列に変換せよ。

In [36]:
!wget https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
!unzip SST-2.zip

--2026-06-03 10:15:36--  https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 99.84.118.67, 99.84.118.30, 99.84.118.60, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|99.84.118.67|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7439277 (7.1M) [application/zip]
Saving to: ‘SST-2.zip.1’

SST-2.zip.1         100%[===================>]   7.09M  40.5MB/s    in 0.2s    

2026-06-03 10:15:36 (40.5 MB/s) - ‘SST-2.zip.1’ saved [7439277/7439277]

Archive:  SST-2.zip
replace SST-2/dev.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [37]:
!head SST-2/dev.tsv

sentence	label
it 's a charming and often affecting journey . 	1
unflinchingly bleak and desperate 	0
allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmmaker . 	1
the acting , costumes , music , cinematography and sound are all astounding given the production 's austere locales . 	1
it 's slow -- very , very slow . 	0
although laced with humor and a few fanciful touches , the film is a refreshingly serious look at young women . 	1
a sometimes tedious film . 	0
or doing last year 's taxes with your ex-wife . 	0
you do n't have to know about music to appreciate the film 's easygoing blend of comedy and romance . 	1


In [38]:
import csv

In [39]:
def load_tsv(filepath):
    """
    SST-2 の TSV を読み込み、sentences と labels のリストを返す。
    ヘッダ行: sentence \t label
    """
    sentences, labels = [], []
    with open(filepath, encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            sentences.append(row["sentence"])
            labels.append(int(row["label"]))   # 0=負(negative), 1=正(positive)
    return sentences, labels

In [40]:
train_sentences, train_labels = load_tsv("SST-2/train.tsv")
dev_sentences,   dev_labels   = load_tsv("SST-2/dev.tsv")
print(f"訓練セット: {len(train_sentences):,} 件")
print(f"開発セット: {len(dev_sentences):,} 件")

訓練セット: 67,349 件
開発セット: 872 件


In [41]:
#tokenize
train_encodings = tokenizer(train_sentences, padding=True, truncation=True, return_tensors="pt")
dev_encodings = tokenizer(dev_sentences, padding=True, truncation=True, return_tensors="pt")

## 86. ミニバッチの作成

85で読み込んだ訓練データの一部（例えば冒頭の4事例）に対して、パディングなどの処理を行い、トークン列の長さを揃えてミニバッチを構成せよ。

In [42]:
import torch

# 訓練データの先頭4件
batch_sentences = train_sentences[:4]
batch_labels = train_labels[:4]

# トークン化 + パディング
batch_encodings = tokenizer(
    batch_sentences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# ミニバッチ
input_ids = batch_encodings["input_ids"]
attention_mask = batch_encodings["attention_mask"]
labels = torch.tensor(batch_labels)

print("input_ids:")
print(input_ids)

print("\nattention_mask:")
print(attention_mask)

print("\nlabels:")
print(labels)

print("\ninput_ids shape:", input_ids.shape)
print("attention_mask shape:", attention_mask.shape)
print("labels shape:", labels.shape)

input_ids:
tensor([[  101,  5342,  2047,  3595,  8496,  2013,  1996, 18643,  3197,   102,
             0,     0,     0,     0,     0],
        [  101,  3397,  2053, 15966,  1010,  2069,  4450,  2098, 18201,  2015,
           102,     0,     0,     0,     0],
        [  101,  2008,  7459,  2049,  3494,  1998, 10639,  2015,  2242,  2738,
          3376,  2055,  2529,  3267,   102],
        [  101,  3464, 12580,  8510,  2000,  3961,  1996,  2168,  2802,   102,
             0,     0,     0,     0,     0]])

attention_mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])

labels:
tensor([0, 0, 1, 0])

input_ids shape: torch.Size([4, 15])
attention_mask shape: torch.Size([4, 15])
labels shape: torch.Size([4])


## 87. ファインチューニング

訓練セットを用い、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

In [43]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset
import numpy as np
from sklearn.metrics import accuracy_score

# Datasetクラス
class SST2Dataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SST2Dataset(train_encodings, train_labels)
dev_dataset = SST2Dataset(dev_encodings, dev_labels)

# 分類モデル
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# 評価指標
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

# 学習設定
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

# ファインチューニング
trainer.train()

# 評価
results = trainer.evaluate()
print(results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 88. 極性分析

問題87でファインチューニングされたモデルを用いて、以下の文の極性を予測せよ。

- "The movie was full of incomprehensibilities."
- "The movie was full of fun."
- "The movie was full of excitement."
- "The movie was full of crap."
- "The movie was full of rubbish."


In [ ]:
import torch
import torch.nn.functional as F

texts = [
    "The movie was full of incomprehensibilities.",
    "The movie was full of fun.",
    "The movie was full of excitement.",
    "The movie was full of crap.",
    "The movie was full of rubbish.",
]

model = trainer.model
model.eval()

encodings = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

device = model.device
encodings = {k: v.to(device) for k, v in encodings.items()}

with torch.no_grad():
    outputs = model(**encodings)
    probs = F.softmax(outputs.logits, dim=1)
    preds = torch.argmax(probs, dim=1)

for text, pred, prob in zip(texts, preds, probs):
    label = "positive" if pred.item() == 1 else "negative"
    print(text)
    print(label, prob.cpu().numpy())

## 89. アーキテクチャの変更

問題87とは異なるアーキテクチャ（例えば[CLS]トークンを用いるか、各トークンの最大値プーリングを用いるなど）の分類モデルを設計し、事前学習済みモデルを極性分析タスク向けにファインチューニングせよ。検証セット上でファインチューニングされたモデルの正解率を計測せよ。

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score
import numpy as np

class BertCLSClassifier(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_labels=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        cls_vec = outputs.last_hidden_state[:, 0, :]  # [CLS]
        logits = self.classifier(self.dropout(cls_vec))

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

In [ ]:
model89 = BertCLSClassifier()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

training_args89 = TrainingArguments(
    output_dir="./results89",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
)

trainer89 = Trainer(
    model=model89,
    args=training_args89,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

trainer89.train()
results89 = trainer89.evaluate()
print(results89)